<a href="https://colab.research.google.com/github/Teixeiras15-collab/Atividades/blob/main/Visualiza%C3%A7%C3%A3o_Geoespacial_com_Folium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Configuração do Ambiente e Base de Dados**
Execute o bloco abaixo no seu ambiente (como o Google Colab) para importar as bibliotecas e gerar o DataFrame com as coordenadas geográficas simuladas:


In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)

### Parte 1: Inicialização e Marcadores Básicos

In [2]:
# Calcular a coordenada média para centralizar o mapa
lat_media = df_mapa['latitude'].mean()
lon_media = df_mapa['longitude'].mean()

# Criar o mapa base
mapa_basico = folium.Map(location=[lat_media, lon_media], zoom_start=12, tiles='OpenStreetMap')

# Iterar sobre as 5 primeiras linhas do DataFrame e adicionar marcadores
for index, row in df_mapa.head(5).iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Tipo: {row['tipo']}<br>Valor: R$ {row['valor_venda']:.2f}"
    ).add_to(mapa_basico)

# Exibir o mapa
display(mapa_basico)

### Parte 2: Customização Visual com Marcadores Circulares

In [4]:
# Criar um novo mapa base para os marcadores circulares
mapa_circulos = folium.Map(location=[lat_media, lon_media], zoom_start=12, tiles='OpenStreetMap')

# Definir as cores para cada cidade
cores_cidade = {
    'Nova Iguaçu': 'blue',
    'Queimados': 'orange'
}

# Iterar sobre todas as linhas do DataFrame e adicionar CircleMarkers
for index, row in df_mapa.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,  # Raio fixo em 8 pixels
        color='black', # Cor da borda
        fill=True,
        fill_color=cores_cidade[row['cidade']], # Cor de preenchimento baseada na cidade
        fill_opacity=0.7,
        tooltip="Clique para detalhes" # Tooltip ao passar o mouse
    ).add_to(mapa_circulos)

# Exibir o mapa com os marcadores circulares
display(mapa_circulos)

### Parte 3: Agrupamento Inteligente (Clustering)

In [5]:
# Criar um terceiro mapa base para o agrupamento
mapa_cluster = folium.Map(location=[lat_media, lon_media], zoom_start=12, tiles='OpenStreetMap')

# Instanciar o objeto MarkerCluster
marker_cluster = MarkerCluster().add_to(mapa_cluster)

# Definir as cores dos ícones para cada tipo de imóvel
cores_tipo = {
    'Casa': 'green',
    'Apartamento': 'blue',
    'Terreno': 'gray'
}

# Iterar sobre todas as linhas do DataFrame e adicionar os imóveis ao cluster com ícones customizados
for index, row in df_mapa.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"ID: {row['id_imovel']}<br>Tipo: {row['tipo']}<br>Cidade: {row['cidade']}<br>Valor: R$ {row['valor_venda']:.2f}",
        icon=folium.Icon(color=cores_tipo[row['tipo']], icon='home', prefix='fa') # Ícone customizado
    ).add_to(marker_cluster)

# Salvar o mapa final em um arquivo HTML
mapa_cluster.save('mapa_imoveis_baixada.html')

# Exibir o mapa (opcional, pois já foi salvo)
display(mapa_cluster)
print("Mapa salvo como 'mapa_imoveis_baixada.html'")

Mapa salvo como 'mapa_imoveis_baixada.html'
